In [0]:

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/court_data/NDAP_REPORT_7150.csv")

print("Shape:", df_raw.count(), "rows,", len(df_raw.columns), "columns")
print("\nColumns:", df_raw.columns)
df_raw.show(5)

In [0]:

for col in df_raw.columns:
    print(col)

In [0]:
from pyspark.sql.functions import col

# Clean and rename columns
df_clean = df_raw.select(
    col("State").alias("state"),
    col("District").alias("district"),
    col("Year").alias("year"),
    col("District and taluk court case type").alias("case_type"),
    col("Pending cases").cast("int").alias("total_pending"),
    col("Pending cases for a period of 0 to 1 years").cast("int").alias("pending_0_1yr"),
    col("Pending cases for a period of 1 to 3 years").cast("int").alias("pending_1_3yr"),
    col("Pending cases for a period of 5 to 10 years").cast("int").alias("pending_5_10yr"),
    col("Pending cases for a period of 10 to 20 years").cast("int").alias("pending_10_20yr"),
    col("Pending cases over 30 years").cast("int").alias("pending_30yr_plus"),
    col("Pending appeal cases").cast("int").alias("appeal_cases"),
    col("Original pending cases").cast("int").alias("original_pending")
).dropna(subset=["state", "district", "total_pending"])

print("Clean rows:", df_clean.count())
display(df_clean.limit(5))

In [0]:

df_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("court_cases_delta")

print("✅ Delta Lake table created!")

# Show history — time travel feature
spark.sql("DESCRIBE HISTORY court_cases_delta").show(5, truncate=False)

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import col, round as spark_round

# Create target variable: estimated years to clear backlog
# Logic: weight older cases more heavily
df_ml = df_clean.withColumn(
    "severity_score",
    (col("pending_5_10yr") * 7 + 
     col("pending_10_20yr") * 15 + 
     col("pending_30yr_plus") * 35) / (col("total_pending") + 1)
).withColumn(
    "years_to_clear",
    spark_round((col("total_pending") / (col("total_pending") - col("pending_0_1yr") + 1)), 2)
).dropna()

# Assemble features
assembler = VectorAssembler(
    inputCols=["total_pending", "pending_0_1yr", "pending_1_3yr", 
               "pending_5_10yr", "pending_10_20yr", "pending_30yr_plus", "severity_score"],
    outputCol="features"
)
df_assembled = assembler.transform(df_ml).select("features", "years_to_clear", "state", "district")

# Train/test split
train, test = df_assembled.randomSplit([0.8, 0.2], seed=42)

# Train model
lr = LinearRegression(featuresCol="features", labelCol="years_to_clear", maxIter=10)
model = lr.fit(train)

print(f"✅ Model trained!")
print(f"R² Score: {round(model.summary.r2, 4)}")
print(f"RMSE: {round(model.summary.rootMeanSquaredError, 4)}")

In [0]:
from pyspark.sql.functions import sum as spark_sum, avg, desc

# Top 10 worst backlog states
print("=== TOP 10 STATES BY BACKLOG ===")
spark.sql("""
    SELECT state, 
           SUM(total_pending) as total_pending_cases,
           SUM(pending_30yr_plus) as cases_over_30_years
    FROM court_cases_delta
    GROUP BY state
    ORDER BY total_pending_cases DESC
    LIMIT 10
""").show()

# Criminal vs Civil breakdown
print("=== CRIMINAL VS CIVIL ===")
spark.sql("""
    SELECT case_type,
           SUM(total_pending) as total,
           AVG(total_pending) as avg_per_district
    FROM court_cases_delta
    GROUP BY case_type
    ORDER BY total DESC
""").show()

# Year-wise trend
print("=== YEAR WISE TREND ===")
spark.sql("""
    SELECT year,
           SUM(total_pending) as total_pending
    FROM court_cases_delta
    GROUP BY year
    ORDER BY year
""").show()

In [0]:
import mlflow

# Fixed: no model saving, just log metrics
with mlflow.start_run(run_name="court_backlog_predictor"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("train_size", "80%")
    mlflow.log_metric("r2_score", model.summary.r2)
    mlflow.log_metric("rmse", model.summary.rootMeanSquaredError)

print("✅ Logged to MLflow!")
print(f"R² Score: {round(model.summary.r2, 4)}")
print(f"RMSE: {round(model.summary.rootMeanSquaredError, 4)}")

In [0]:
from pyspark.sql.functions import sum as spark_sum, avg, desc

# Top 10 worst backlog states
print("=== TOP 10 STATES BY BACKLOG ===")
spark.sql("""
    SELECT state, 
           SUM(total_pending) as total_pending_cases,
           SUM(pending_30yr_plus) as cases_over_30_years
    FROM court_cases_delta
    GROUP BY state
    ORDER BY total_pending_cases DESC
    LIMIT 10
""").show()

# Criminal vs Civil breakdown
print("=== CRIMINAL VS CIVIL ===")
spark.sql("""
    SELECT case_type,
           SUM(total_pending) as total,
           AVG(total_pending) as avg_per_district
    FROM court_cases_delta
    GROUP BY case_type
    ORDER BY total DESC
""").show()

# Year-wise trend
print("=== YEAR WISE TREND ===")
spark.sql("""
    SELECT year,
           SUM(total_pending) as total_pending
    FROM court_cases_delta
    GROUP BY year
    ORDER BY year
""").show()

In [0]:
# Show predictions — this is your live demo moment
predictions = model.transform(test)

print("=== SAMPLE PREDICTIONS ===")
predictions.select("state", "district", "years_to_clear", "prediction") \
    .withColumnRenamed("prediction", "predicted_years") \
    .show(10)

# Time travel demo — judges will love this
print("\n=== DELTA LAKE TIME TRAVEL ===")
spark.sql("DESCRIBE HISTORY court_cases_delta").select("version", "timestamp", "operation").show()

In [0]:
spark.sql("""
    SELECT state,
           SUM(total_pending) as current_backlog,
           ROUND(SUM(total_pending) * 0.7, 0) as backlog_if_30pct_more_judges,
           ROUND(try_divide(SUM(pending_30yr_plus), SUM(total_pending)) * 100, 2) as pct_cases_over_30yrs
    FROM court_cases_delta
    GROUP BY state
    ORDER BY current_backlog DESC
    LIMIT 10
""").show()

In [0]:
# Show data coverage — proves dataset quality
print("=== DATASET COVERAGE ===")
spark.sql("""
    SELECT 
        COUNT(DISTINCT state) as states_covered,
        COUNT(DISTINCT district) as districts_covered,
        MIN(year) as data_from,
        MAX(year) as data_to,
        SUM(total_pending) as total_cases_analyzed
    FROM court_cases_delta
""").show()